# Cascade stage1a + stage1b — dual-GPU parallel training

Trains both binary relevance gates (`buyin_relevance` on GPU0, `stance_relevance` on GPU1)
**concurrently** in one Kaggle T4 x2 session, instead of sequentially.
Each model gets its own full T4, own Trainer, own checkpoint — no DataParallel splitting,
no communication overhead between the two GPUs. Replaces having to run two separate notebook sessions.

In [ ]:
!pip install -q -U transformers datasets scikit-learn

In [ ]:
import glob, json, random, shutil, threading, traceback
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score, cohen_kappa_score, classification_report,
    confusion_matrix, precision_recall_fscore_support,
)
from torch.utils.data import Dataset
from transformers import (
    AutoTokenizer, AutoModelForSequenceClassification,
    TrainingArguments, Trainer, EarlyStoppingCallback, DataCollatorWithPadding,
)

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

N_GPU = torch.cuda.device_count()
print(f"GPU count: {N_GPU}")
for i in range(N_GPU):
    print(f"  cuda:{i} -> {torch.cuda.get_device_name(i)}")
assert N_GPU >= 2, "This notebook expects a T4 x2 (or similar dual-GPU) session."

In [ ]:
def find_input_file(*names):
    for name in names:
        for pattern in [f"/kaggle/input/**/{name}", f"/kaggle/input/datasets/kevinnchan/**/{name}"]:
            matches = glob.glob(pattern, recursive=True)
            if matches:
                return Path(matches[0])
        if Path(name).exists():
            return Path(name)
    raise FileNotFoundError(f"Cannot find any of {names}")


class CascadeDataset(Dataset):
    def __init__(self, encodings, label_ids, weights=None):
        self.encodings = encodings
        self.labels    = label_ids
        self.weights   = weights if weights is not None else [1.0] * len(label_ids)

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        item = {k: torch.tensor(v[idx]) for k, v in self.encodings.items()}
        item["labels"]  = torch.tensor(self.labels[idx], dtype=torch.long)
        item["weights"] = torch.tensor(self.weights[idx], dtype=torch.float)
        return item


class WeightedCollator(DataCollatorWithPadding):
    def __call__(self, features):
        labels  = torch.tensor([f.pop("labels")           for f in features], dtype=torch.long)
        weights = torch.tensor([f.pop("weights", 1.0)     for f in features], dtype=torch.float)
        batch   = super().__call__(features)
        batch["labels"]  = labels
        batch["weights"] = weights
        return batch


class WeightedTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels  = inputs.pop("labels")
        weights = inputs.pop("weights")
        outputs = model(**inputs)
        loss_fn = nn.CrossEntropyLoss(reduction="none")
        loss    = (loss_fn(outputs.logits, labels) * weights).mean()
        return (loss, outputs) if return_outputs else loss


def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {
        "kappa":    cohen_kappa_score(labels, preds),
        "accuracy": accuracy_score(labels, preds),
    }

In [ ]:
# ── Per-stage config — same hyperparameters as the original single-GPU notebooks ──
MODEL_NAME = "zanelim/singbert-large-sg"
MAX_LEN    = 128
BATCH_SIZE = 32
EPOCHS     = 6
LR         = 2e-05
GRAD_ACCUM = 2
VAL_FRAC   = 0.15

STAGE_CONFIGS = {
    "stage1a": dict(
        gpu_id     = 0,
        axis       = "buyin_relevance",
        train_csv  = "stage1a_train.csv",
        id2label   = {"0": "not_relevant", "1": "relevant"},
        label2id   = {"not_relevant": 0, "relevant": 1},
        gold_map   = {"committed": "relevant", "uncommitted": "relevant", "neutral": "not_relevant"},
        gold_col   = "human_label",
    ),
    "stage1b": dict(
        gpu_id     = 1,
        axis       = "stance_relevance",
        train_csv  = "stage1b_train.csv",
        id2label   = {"0": "no_stance", "1": "has_stance"},
        label2id   = {"no_stance": 0, "has_stance": 1},
        gold_map   = {"supportive": "has_stance", "critical": "has_stance", "neutral": "no_stance"},
        gold_col   = "human_stance",
    ),
}

In [ ]:
def train_stage(stage_name, cfg, results, errors):
    """Runs entirely on one GPU (cfg['gpu_id']). Called inside its own thread."""
    try:
        gpu_id  = cfg["gpu_id"]
        device  = torch.device(f"cuda:{gpu_id}")
        torch.cuda.set_device(gpu_id)  # thread-local: pins this thread's default CUDA device

        id2label   = cfg["id2label"]
        label2id   = cfg["label2id"]
        num_labels = len(id2label)
        output_dir = Path(f"/kaggle/working/{stage_name}")
        output_dir.mkdir(parents=True, exist_ok=True)

        print(f"[{stage_name}] device={device}  axis={cfg['axis']}")

        # ── Load + prep training data ──
        train_path = find_input_file(cfg["train_csv"])
        df = pd.read_csv(train_path)
        df = df[df["label"].notna()].copy()
        if df["label"].dtype == object:
            df["label_id"] = df["label"].map(label2id)
        else:
            df["label_id"] = df["label"].astype(int)
        df = df[df["label_id"].notna()].copy()
        df["label_id"] = df["label_id"].astype(int)
        df["weight"]   = df["weight"].fillna(1.0).astype(float)
        df["text"]     = df["text"].fillna("").astype(str).str.strip()
        df = df[df["text"].str.len() > 5].reset_index(drop=True)

        train_df, val_df = train_test_split(
            df, test_size=VAL_FRAC, random_state=SEED, stratify=df["label_id"]
        )
        print(f"[{stage_name}] Train: {len(train_df):,}  Val: {len(val_df):,}")

        # ── Tokenise ──
        tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
        train_enc = tokenizer(train_df["text"].tolist(), truncation=True, max_length=MAX_LEN, padding=False)
        val_enc   = tokenizer(val_df["text"].tolist(),   truncation=True, max_length=MAX_LEN, padding=False)
        train_dataset = CascadeDataset(train_enc, train_df["label_id"].tolist(), train_df["weight"].tolist())
        val_dataset   = CascadeDataset(val_enc,   val_df["label_id"].tolist(),   [1.0] * len(val_df))

        # ── Model + Trainer, pinned to this GPU only ──
        model = AutoModelForSequenceClassification.from_pretrained(
            MODEL_NAME, num_labels=num_labels,
            id2label={int(k): v for k, v in id2label.items()}, label2id=label2id,
            ignore_mismatched_sizes=True,
        )
        model.to(device)

        training_args = TrainingArguments(
            output_dir                  = str(output_dir),
            num_train_epochs            = EPOCHS,
            per_device_train_batch_size = BATCH_SIZE,
            per_device_eval_batch_size  = BATCH_SIZE * 2,
            gradient_accumulation_steps = GRAD_ACCUM,
            learning_rate               = LR,
            weight_decay                = 0.01,
            warmup_ratio                = 0.06,
            eval_strategy               = "epoch",
            save_strategy               = "epoch",
            save_total_limit            = 1,
            load_best_model_at_end      = True,
            metric_for_best_model       = "kappa",
            greater_is_better           = True,
            fp16                        = True,
            dataloader_num_workers      = 2,
            report_to                   = "none",
        )
        # Force single-GPU path: without this, HF Trainer sees 2 visible GPUs and
        # wraps the model in DataParallel across BOTH, which fights with the other
        # thread's model for the same two devices. Pinning n_gpu=1 keeps this
        # thread's model entirely on the GPU we already moved it to.
        training_args._n_gpu = 1

        trainer = WeightedTrainer(
            model           = model,
            args            = training_args,
            train_dataset   = train_dataset,
            eval_dataset    = val_dataset,
            compute_metrics = compute_metrics,
            callbacks       = [EarlyStoppingCallback(early_stopping_patience=2)],
            data_collator   = WeightedCollator(tokenizer),
        )

        trainer.train()
        print(f"[{stage_name}] Best val kappa: {trainer.state.best_metric:.4f}")

        # ── Evaluate on gold testset ──
        test_path = find_input_file("commitment_testset.parquet")
        test_df   = pd.read_parquet(test_path)
        gold_col  = cfg["gold_col"]
        test_df[gold_col] = test_df[gold_col].map(cfg["gold_map"])
        test_df = test_df[test_df[gold_col].isin(label2id)].reset_index(drop=True)

        test_enc = tokenizer(test_df["text"].tolist(), truncation=True, max_length=MAX_LEN, padding=False)
        test_labels = test_df[gold_col].map(label2id).tolist()
        test_dataset = CascadeDataset(test_enc, test_labels)

        out   = trainer.predict(test_dataset)
        preds = np.argmax(out.predictions, axis=-1)
        id2label_int = {int(k): v for k, v in id2label.items()}
        preds_named = [id2label_int[int(p)] for p in preds]
        true_named  = test_df[gold_col].tolist()

        print(f"\n{'='*72}\n  {stage_name} — {cfg['axis']} — Testset Evaluation\n{'='*72}")
        print(f"  Accuracy: {accuracy_score(test_labels, preds):.1%}")
        print(f"  Kappa:    {cohen_kappa_score(test_labels, preds):.3f}")
        print(classification_report(true_named, preds_named, digits=3))

        eval_df = test_df.copy()
        eval_df["pred_label"] = preds_named
        eval_df.to_csv(output_dir / f"testset_eval_{stage_name}.csv", index=False)

        # ── Save model ──
        model_out = output_dir / "best_model"
        trainer.save_model(str(model_out))
        tokenizer.save_pretrained(str(model_out))
        with open(model_out / "id2label.json", "w") as f:
            json.dump({"axis": cfg["axis"], "id2label": id2label, "label2id": label2id}, f, indent=2)
        zip_path = str(output_dir.parent / f"cascade_{cfg['axis']}")
        shutil.make_archive(zip_path, "zip", str(model_out))
        print(f"[{stage_name}] Saved + zipped -> {zip_path}.zip")

        results[stage_name] = trainer.state.best_metric
    except Exception as e:
        errors.append((stage_name, traceback.format_exc()))
        print(f"[{stage_name}] FAILED: {e}")

In [ ]:
# ── Launch both trainings in parallel, one per GPU ──
results, errors = {}, []
threads = [
    threading.Thread(target=train_stage, args=(name, cfg, results, errors))
    for name, cfg in STAGE_CONFIGS.items()
]
for t in threads: t.start()
for t in threads: t.join()

print("\n" + "="*72)
print("DONE")
print("Best kappas:", results)
if errors:
    print(f"\n{len(errors)} stage(s) FAILED:")
    for name, tb in errors:
        print(f"\n--- {name} ---\n{tb}")